In [1]:
import os
import sys
import session_info
import geopandas as gpd
import numpy as np
import scipy
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import matplotlib.pyplot as plt
import mpl_toolkits
import rtree
from shapely.geometry import Polygon

"""
Reads CD113 census data
Only uses lower 48 states
Updates percison and projection
Calcualtes district border overlapp with coastline and with state border
Calculates Polsby Popper
Outputs a std data format for compiling with other years

"""

Shapepath = "Census data/"
Dataoutpath ="Process data/"

Projection = "ESRI:102008"
#session_info.show()
print( "Libraries loaded")

Libraries loaded


In [2]:
def Border_clip( Geodf, Borderdf):
    """
    Takes a GeoPandas dataframe and clips is borders to the Borders in Borderdf
    Assumes GeoPandas dataframe has rows delineated by GEOID
    Cleans up any non polygon residuals
    Removes any remaining with less than 100,000 sq meters in area
    """
    Districts_clipped = gpd.clip( Geodf, Borderdf, keep_geom_type=False).copy()
    Allgeoms = Districts_clipped.explode()
    Allpolys = Allgeoms[ Allgeoms.geometry.geom_type=="Polygon"].copy()
    Allpolys['Area'] = Allpolys.area
    Allpolys = Allpolys[ Allpolys['Area'] >100000]
    DistrictsM = Allpolys.dissolve(by='GEOID', method='coverage').reset_index()
    DistrictsM.drop(['Area'], axis=1, inplace=True)
    return( DistrictsM)

print( "Border_clip function defined")
    

Border_clip function defined


In [3]:
def Geo_check( Geodf ) :
    """
    Analyze the composition of the geo data frame
    Looks for differing geometry types
    """
    ### Count types of geometries found pre clipping
    geometry_types = {'Point': 0, 'LineString': 0, 'Polygon': 0, 'MultiPolygon': 0, 'GeometryCollection':0}

    for geom in Geodf['geometry']:
        geometry_types[geom.geom_type] += 1 

    print("Points:", geometry_types['Point'])
    print("Lines:", geometry_types['LineString'])
    print("Polygons:", geometry_types['Polygon'])
    print("MultiPolygons:", geometry_types['MultiPolygon'])
    print("GeometryCollections:", geometry_types['GeometryCollection'])
    print("Total Polys:", len( Geodf.explode()))
    print( "Total rows:", len(Geodf) )
    print( "Min area polygon:", int( min( Geodf.explode().area)))
    return
    
print( "Geo_check function defined")

Geo_check function defined


In [4]:
def Polsby(Geometry):
    """
    Calculates the Polsby Poppper compactness ratio of a given geometry. 
    Uses an area weighted average if more than one polygon.
    Uses only the exterior ring of each polgon
    Assumes coordinates have already been projected     
    Args: 
        Geometry (shapely geometry): A Shapely geometry object (e.g., Polygon, MultiPolygon). 
        
    Returns:
        float: The PolsbyPopper score
    """
    Geos = gpd.GeoDataFrame( geometry=[Geometry], crs=Projection)
    #Darea = Geos["geometry"].area
    Geos = Geos.explode( )
    Geos['geometry'] = Geos['geometry'].apply(lambda p: Polygon(p.exterior.coords)) ## make polygon out of exterior corrdinates
    Geos['Perimeter2'] = Geos["geometry"].length**2
    Geos["Area"] = Geos["geometry"].area
    Darea = Geos['Area'].sum()
    Geos["PolsbyI"] = (Geos["Area"] * 4 * 3.14159) /  Geos["Perimeter2"] 
    Geos["PolsbyW"] = Geos["PolsbyI"] * Geos["Area"] / Darea
    Polsby = Geos["PolsbyW"].sum() 
    
    return Polsby
print( "Polsby function defined")   

Polsby function defined


In [5]:
def DSTborder( Geodf) :
    """
    Takes a geopandas dataframe with one states districts
    Assumes Projection is defined globally
    Dataframe has GEOID for each row
    Combines districts to form state
    Finds state border by difference from a buffered state border
    Finds intersection of districts border with state border
    Calcs length of district border that is also on state border DST
    Return dataframe with GEOID and DST
    """
    STunion = Geodf.geometry.union_all()        ### create state from union of all districts
    STall = gpd.GeoDataFrame( geometry=[STunion], crs=Projection) 
    STminus = STall.buffer(-1 )                   ### shrink starte by one meter all around    
    Border = STall.difference( STminus)            ### create one meter wide perimeter inside of state
    Border = gpd.GeoDataFrame(geometry=Border)
    DSTborder = gpd.overlay(Geodf, Border, how='intersection' )
    DSTborder["DSTperim"] =  DSTborder["geometry"].length/2 
    DSTborder["DSTperim"] =  [int(x) for x in  DSTborder["DSTperim"]  ] 
    DSTborder.drop(['geometry'], axis=1, inplace=True)
    return( DSTborder[ ['GEOID', 'DSTperim'] ] )

print("DSTborder defined")

DSTborder defined


In [6]:
def CSTborder( Geodf, Coastdf ):
    """
    Takes a geopandas dataframe with districts
    Assumes all geopandas data frame have same Projection 
    Dataframe has GEOID for each row
    Finds intersection of districts with coastline
    Calcs length of district border that is also on coastline  CST
    Return dataframe with GEOID and CST
     """
    GeodfBuf = Geodf.copy()
    Coastal_Intersect = gpd.overlay( GeodfBuf, Coastdf, how='intersection', keep_geom_type=False )
    Coastal_Intersect['Coast_len'] = Coastal_Intersect.length
    Coastal_Districts = Coastal_Intersect.groupby( Coastal_Intersect["GEOID"]).agg( {'Coast_len':'sum'})
    Coastal_Districts['Coast_len'] = [ int(x) for x in Coastal_Districts['Coast_len'] ]
    return( Coastal_Districts)

print("CSTborder defined")

CSTborder defined


In [14]:
##### Load US boundary file for clipping
File = 'cb_2023_us_nation_20m'
Shapedatafile = Shapepath + File + ".shp"
RawShapes = gpd.read_file( Shapedatafile)
USbox = Polygon( [ (-130,20),(-60,20), (-60,50), (-130,50) ] )        ### Contental US box
Shapes = gpd.clip( RawShapes, USbox)
Shapes.to_crs( Projection, inplace=True)
Shapes.geometry = Shapes.geometry.set_precision( 0.01 )
#Shapes.plot()
print( "Loaded ", File," with ", len( Shapes.explode()), " polygons")
#Shapes.head()
US48 = Shapes.copy()

Loaded  cb_2023_us_nation_5m  with  114  polygons


In [8]:
### Load coastline file for coastlline intersection calc
File = "tl_2023_us_coastline"
Shapedatafile = Shapepath + File + ".shp"
Coastline = gpd.read_file(Shapedatafile)
USbox = Polygon( [ (-130,20),(-60,20), (-60,50), (-130,50) ] )        ### Contental US box
Coastline = gpd.clip( Coastline, USbox)
Coastline.to_crs( crs=Projection, inplace=True )
print(  File +" loaded ", len(Coastline)," rows of coastline data")
#Coastline.plot()
#Coastline.head()
#Geo_check( Coastline)

tl_2023_us_coastline loaded  3315  rows of coastline data


In [9]:
#### Get district shape data 
File = 'tl_2013_us_cd113'
Fout = 'CD113'
Shapedatafile = Shapepath + File + ".shp"
Shapeoutfile = Dataoutpath + Fout + ".shp"
RawDistricts = gpd.read_file( Shapedatafile)
RawDistricts.drop( ['NAMELSAD', 'LSAD', 'MTFCC', 'FUNCSTAT', 'ALAND','AWATER', 'INTPTLAT', 'INTPTLON'] , axis=1, inplace=True)
RawDistricts.rename(columns={'CD113FP': 'DISTRICT'}, inplace=True)
RawDistricts.rename(columns={'CDSESSN': 'SESSN'}, inplace=True)
USbox = Polygon( [ (-130,20),(-60,20), (-60,50), (-130,50) ] )        ### Contental US box
RawDistricts = gpd.clip( RawDistricts, USbox)
RawDistricts.to_crs( Projection, inplace=True)
RawDistricts.geometry = RawDistricts.geometry.set_precision(0.01)
print(  File +" loaded ", len(RawDistricts)," rows of data with", len( RawDistricts.explode()), "polygons")

#RawDistricts.plot()
#RawDistricts.head()

tl_2013_us_cd113 loaded  436  rows of data with 459 polygons


In [15]:
### Clip borders to overall US
Districts = Border_clip( RawDistricts, US48 )
#Geo_check( Districts )
print('done clipping')

done clipping


In [14]:
#### Calc district level metrics
Districts['Dperimeter'] = Districts.length
Districts['Dperimeter'] = [int(x) for x in Districts['Dperimeter'] ] 
Districts['Darea'] = Districts.area
Districts['Darea'] = [int(x) for x in Districts['Darea'] ] 
Coast = CSTborder(Districts, Coastline)
Districts = pd.merge( Districts, Coast, on='GEOID', how='left')
Districts['Coast_len'] = Districts['Coast_len'].fillna(0)
Districts['PolsbyW'] = Districts.geometry.apply( Polsby )
print( 'Done with district level metrics')
#Districts.head()

Done with district level metrics


In [15]:
#### Calculate state metrics
STdata = pd.DataFrame(columns=['STATEFP', 'STarea', 'STperim','STpolsby'])
States = Districts[ 'STATEFP'].unique()
#print( "Found ", len(States), ' states')
for STFP in States :
    #print( STFP)
    STdists = Districts[ Districts['STATEFP'] == STFP]
    State = STdists.geometry.union_all()
    Sarea = int( State.area )
    Sperim = int( State.length )
    Spolsby =  Polsby(State)
    STdata.loc[len(STdata)] = { 'STATEFP':STFP, 'STarea':Sarea, 'STperim':Sperim, 'STpolsby':Spolsby }

Districts = pd.merge( Districts, STdata, on='STATEFP', how='left')

print( 'Done state data calcs')
#STdata.head()
#Districts.head()


Done state data calcs


In [16]:
#### Calc district perim on state border for each state - district
DSTdata = pd.DataFrame(columns=['GEOID', 'DSTperim'])
States = Districts[ 'STATEFP'].unique()
for STFP in States :
    #print( STFP )
    DST = DSTborder( Districts[ Districts['STATEFP']==STFP ] )
    DSTdata = pd.concat( [DSTdata, DST] , ignore_index=True )

#DSTdata.head()
Districts = pd.merge( Districts, DSTdata, on='GEOID', how='left')
Districts['DSTperim'] = Districts['DSTperim'].fillna(0)
print( 'DST calc done' )
#Districts.head()

DST calc done


In [17]:
Column_order = [ 'GEOID', 'STATEFP', 'DISTRICT', 'SESSN', 'Dperimeter', 'Darea', 'PolsbyW', 'DSTperim','Coast_len', 'STarea', 'STperim', 'STpolsby', 'geometry' ]
Districts = Districts.reindex(columns=Column_order)
Districts.to_file( Shapeoutfile )
print( 'Done writing ', len( Districts ), 'rows to ', Shapeoutfile)
#Districts.plot()
#Geo_check( Districts )
#Districts.head()

Done writing  436 rows to  Process data/CD113.shp


,GEOID,STATEFP,DISTRICT,SESSN,Dperimeter,Darea,PolsbyW,DSTperim,Coast_len,STarea,STperim,STpolsby,geometry
0,0101,01,01,113,1236873,16168059400,0.132806,552623,152078.0,133946536574,1860712,0.486164,"POLYGON ((788741.17 -974633.34, 788739.11 -974..."
1,0102,01,02,113,1212446,26573596185,0.227161,300869,0.0,133946536574,1860712,0.486164,"POLYGON ((881674.7 -856357.09, 881683.5 -85637..."
2,0103,01,03,113,1072170,20009510042,0.218735,315423,0.0,133946536574,1860712,0.486164,"POLYGON ((847843.12 -673307.11, 847826.24 -673..."
3,0104,01,04,113,1284512,23595740637,0.179707,220405,0.0,133946536574,1860712,0.486164,"POLYGON ((842337.33 -591866.73, 842273.61 -591..."
4,0105,01,05,113,649466,9985458647,0.297485,256996,0.0,133946536574,1860712,0.486164,"POLYGON ((674258.35 -558945.72, 674269.06 -558..."
